In [8]:
# Imports and Setup
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [9]:
# Data Loading - Standard Train/Test Split
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

# Load training data (60,000 images)
trainset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

# Load test data (10,000 images)
testset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=False, transform=transform)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

print(f'Training samples: {len(trainset)}')
print(f'Test samples: {len(testset)}')

Training samples: 60000
Test samples: 10000


In [10]:
# Helper Functions
def train_model(model, trainloader, criterion, optimizer, scheduler=None, epochs=5):
    """Train a model and return training history"""
    model.train()
    train_losses = []

    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(trainloader)
        train_losses.append(epoch_loss)
        print(f'Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}')

        if scheduler:
            scheduler.step()

    return train_losses

def evaluate_model(model, testloader):
    """Evaluate model and return accuracy"""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

In [11]:
# Baseline: MLP (Multi-Layer Perceptron) - From Lesson 3.7 Neural Network & Deep Learning
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.shape[0], -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.log_softmax(x, dim=1)

In [12]:
# Train Baseline MLP
print("Training Baseline MLP...")
mlp_model = MLP().to(device)
mlp_criterion = nn.NLLLoss()
mlp_optimizer = optim.Adam(mlp_model.parameters(), lr=0.003)

mlp_losses = train_model(mlp_model, trainloader, mlp_criterion, mlp_optimizer)
mlp_accuracy = evaluate_model(mlp_model, testloader)
print(f'MLP Accuracy: {mlp_accuracy:.2f}%\n')

Training Baseline MLP...
Epoch 1/5, Loss: 0.2291
Epoch 2/5, Loss: 0.1100
Epoch 3/5, Loss: 0.0872
Epoch 4/5, Loss: 0.0745
Epoch 5/5, Loss: 0.0670
MLP Accuracy: 97.42%



In [13]:
# Experiment 1: Basic CNN - Target: ~98%
class BasicCNN(nn.Module):
    def __init__(self):
        super(BasicCNN, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  # 28x28 -> 28x28
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 14x14 -> 14x14

        # Pooling layer
        self.pool = nn.MaxPool2d(2, 2)  # 28x28 -> 14x14, 14x14 -> 7x7

        # Fully connected layers
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Conv + ReLU + Pool
        x = self.pool(F.relu(self.conv1(x)))  # 28x28 -> 14x14
        x = self.pool(F.relu(self.conv2(x)))  # 14x14 -> 7x7

        # Flatten
        x = x.view(x.size(0), -1)  # 64*7*7 = 3136

        # Fully connected
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [14]:
# Train Basic CNN
print("Training Basic CNN...")
cnn1_model = BasicCNN().to(device)
cnn1_criterion = nn.NLLLoss()
cnn1_optimizer = optim.Adam(cnn1_model.parameters(), lr=0.003)

cnn1_losses = train_model(cnn1_model, trainloader, cnn1_criterion, cnn1_optimizer)
cnn1_accuracy = evaluate_model(cnn1_model, testloader)
print(f'Basic CNN Accuracy: {cnn1_accuracy:.2f}%\n')

Training Basic CNN...
Epoch 1/5, Loss: 0.1258
Epoch 2/5, Loss: 0.0428
Epoch 3/5, Loss: 0.0308
Epoch 4/5, Loss: 0.0236
Epoch 5/5, Loss: 0.0204
Basic CNN Accuracy: 98.93%



In [15]:
# Experiment 2: CNN with BatchNorm and Dropout - Target: ~98.5%
class RegularizedCNN(nn.Module):
    def __init__(self):
        super(RegularizedCNN, self).__init__()
        # Convolutional layers with BatchNorm
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Pooling
        self.pool = nn.MaxPool2d(2, 2)

        # Fully connected with dropout
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Conv + BatchNorm + ReLU + Pool
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully connected with dropout
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [16]:
# Train Regularized CNN
print("Training CNN with BatchNorm and Dropout...")
cnn2_model = RegularizedCNN().to(device)
cnn2_criterion = nn.NLLLoss()
cnn2_optimizer = optim.Adam(cnn2_model.parameters(), lr=0.003)

cnn2_losses = train_model(cnn2_model, trainloader, cnn2_criterion, cnn2_optimizer)
cnn2_accuracy = evaluate_model(cnn2_model, testloader)
print(f'Regularized CNN Accuracy: {cnn2_accuracy:.2f}%\n')

Training CNN with BatchNorm and Dropout...
Epoch 1/5, Loss: 0.1939
Epoch 2/5, Loss: 0.0807
Epoch 3/5, Loss: 0.0593
Epoch 4/5, Loss: 0.0503
Epoch 5/5, Loss: 0.0410
Regularized CNN Accuracy: 99.02%



In [17]:
# Experiment 3: Data Augmentation - Target: ~99%
# Enhanced transforms for training
transform_augmented = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Create augmented training dataset
trainset_aug = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform_augmented)
trainloader_aug = DataLoader(trainset_aug, batch_size=64, shuffle=True)

# Same architecture as Exercise 2
class AugmentedCNN(nn.Module):
    def __init__(self):
        super(AugmentedCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [18]:
# Train CNN with Data Augmentation
print("Training CNN with Data Augmentation...")
cnn3_model = AugmentedCNN().to(device)
cnn3_criterion = nn.NLLLoss()
cnn3_optimizer = optim.Adam(cnn3_model.parameters(), lr=0.003)

cnn3_losses = train_model(cnn3_model, trainloader_aug, cnn3_criterion, cnn3_optimizer)
cnn3_accuracy = evaluate_model(cnn3_model, testloader)
print(f'Augmented CNN Accuracy: {cnn3_accuracy:.2f}%\n')

Training CNN with Data Augmentation...
Epoch 1/5, Loss: 0.5532
Epoch 2/5, Loss: 0.2793
Epoch 3/5, Loss: 0.2163
Epoch 4/5, Loss: 0.1855
Epoch 5/5, Loss: 0.1567
Augmented CNN Accuracy: 99.10%



In [19]:
# Experiment 4: Learning Rate Scheduling - Target: ~99.2%
class ScheduledCNN(nn.Module):
    def __init__(self):
        super(ScheduledCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [20]:
# Train CNN with Learning Rate Scheduling
print("Training CNN with Learning Rate Scheduling...")
cnn4_model = ScheduledCNN().to(device)
cnn4_criterion = nn.NLLLoss()
cnn4_optimizer = optim.Adam(cnn4_model.parameters(), lr=0.01)  # Higher initial LR
cnn4_scheduler = optim.lr_scheduler.StepLR(cnn4_optimizer, step_size=2, gamma=0.7)

cnn4_losses = train_model(cnn4_model, trainloader_aug, cnn4_criterion, cnn4_optimizer, cnn4_scheduler, epochs=20)
cnn4_accuracy = evaluate_model(cnn4_model, testloader)
print(f'Scheduled CNN Accuracy: {cnn4_accuracy:.2f}%\n')
# 1. Move the model back to CPU before saving (best practice for deployment)
cnn4_model.to('cpu')

# 2. Save the state dictionary
torch.save(cnn4_model.state_dict(), 'mnist_cnn_weights.pth')
print("Model weights saved successfully!")

Training CNN with Learning Rate Scheduling...
Epoch 1/20, Loss: 1.3794
Epoch 2/20, Loss: 0.5508
Epoch 3/20, Loss: 0.3172
Epoch 4/20, Loss: 0.2690
Epoch 5/20, Loss: 0.2187
Epoch 6/20, Loss: 0.1766
Epoch 7/20, Loss: 0.1573
Epoch 8/20, Loss: 0.1465
Epoch 9/20, Loss: 0.1358
Epoch 10/20, Loss: 0.1359
Epoch 11/20, Loss: 0.1204
Epoch 12/20, Loss: 0.1116
Epoch 13/20, Loss: 0.0979
Epoch 14/20, Loss: 0.0975
Epoch 15/20, Loss: 0.0946
Epoch 16/20, Loss: 0.0902
Epoch 17/20, Loss: 0.0889
Epoch 18/20, Loss: 0.0878
Epoch 19/20, Loss: 0.0843
Epoch 20/20, Loss: 0.0877
Scheduled CNN Accuracy: 99.35%

Model weights saved successfully!


In [21]:
# Experiment 5: Advanced ResNet-style Architecture - Target: ~99.3%+
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class AdvancedCNN(nn.Module):
    def __init__(self):
        super(AdvancedCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)

        # Residual blocks
        self.layer1 = ResidualBlock(32, 64, stride=2)  # 28x28 -> 14x14
        self.layer2 = ResidualBlock(64, 128, stride=2)  # 14x14 -> 7x7

        # Global average pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Classifier
        self.fc = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return F.log_softmax(x, dim=1)

In [22]:
# Train Advanced ResNet-style CNN
print("Training Advanced ResNet-style CNN...")
cnn5_model = AdvancedCNN().to(device)
cnn5_criterion = nn.NLLLoss()
cnn5_optimizer = optim.Adam(cnn5_model.parameters(), lr=0.001)
cnn5_scheduler = optim.lr_scheduler.StepLR(cnn5_optimizer, step_size=2, gamma=0.8)

cnn5_losses = train_model(cnn5_model, trainloader_aug, cnn5_criterion, cnn5_optimizer, cnn5_scheduler, epochs=20)
cnn5_accuracy = evaluate_model(cnn5_model, testloader)
print(f'Advanced CNN Accuracy: {cnn5_accuracy:.2f}%\n')

# 1. Move the model back to CPU before saving (best practice for deployment)
cnn5_model.to('cpu')

# 2. Save the state dictionary
torch.save(cnn5_model.state_dict(), 'mnist_advanced_cnn_weights.pth')
print("Model weights saved successfully!")

Training Advanced ResNet-style CNN...
Epoch 1/20, Loss: 0.2434
Epoch 2/20, Loss: 0.0621
Epoch 3/20, Loss: 0.0461
Epoch 4/20, Loss: 0.0430
Epoch 5/20, Loss: 0.0350
Epoch 6/20, Loss: 0.0334
Epoch 7/20, Loss: 0.0282
Epoch 8/20, Loss: 0.0272
Epoch 9/20, Loss: 0.0239
Epoch 10/20, Loss: 0.0229
Epoch 11/20, Loss: 0.0199
Epoch 12/20, Loss: 0.0205
Epoch 13/20, Loss: 0.0166
Epoch 14/20, Loss: 0.0170
Epoch 15/20, Loss: 0.0158
Epoch 16/20, Loss: 0.0151
Epoch 17/20, Loss: 0.0140
Epoch 18/20, Loss: 0.0139
Epoch 19/20, Loss: 0.0131
Epoch 20/20, Loss: 0.0123
Advanced CNN Accuracy: 99.58%

Model weights saved successfully!


In [23]:
# Results Summary
print("="*50)
print("MNIST Classification Results Summary")
print("="*50)
print(f"Baseline MLP:              {mlp_accuracy:.2f}%")
print(f"Basic CNN:                 {cnn1_accuracy:.2f}%")
print(f"CNN + BatchNorm/Dropout:   {cnn2_accuracy:.2f}%")
print(f"CNN + Data Augmentation:   {cnn3_accuracy:.2f}%")
print(f"CNN + LR Scheduling:       {cnn4_accuracy:.2f}%")
print(f"Advanced ResNet-style:     {cnn5_accuracy:.2f}%")
print("="*50)

MNIST Classification Results Summary
Baseline MLP:              97.42%
Basic CNN:                 98.93%
CNN + BatchNorm/Dropout:   99.02%
CNN + Data Augmentation:   99.10%
CNN + LR Scheduling:       99.35%
Advanced ResNet-style:     99.58%


In [24]:
# Experiment 6: Deeper CNN Architecture - Target: 99.5-99.6%
class DeeperCNN(nn.Module):
    def __init__(self):
        super(DeeperCNN, self).__init__()
        # More convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(256, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # First block
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 28x28 -> 14x14

        # Second block
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))  # 14x14 -> 7x7

        # Global average pooling
        x = self.adaptive_pool(x)  # 7x7 -> 1x1
        x = x.view(x.size(0), -1)

        # Classifier
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [25]:
# Train Deeper CNN
print("Training Deeper CNN...")
cnn6_model = DeeperCNN().to(device)
cnn6_criterion = nn.NLLLoss()
cnn6_optimizer = optim.Adam(cnn6_model.parameters(), lr=0.001)
cnn6_scheduler = optim.lr_scheduler.StepLR(cnn6_optimizer, step_size=3, gamma=0.7)

cnn6_losses = train_model(cnn6_model, trainloader_aug, cnn6_criterion, cnn6_optimizer, cnn6_scheduler)
cnn6_accuracy = evaluate_model(cnn6_model, testloader)
print(f'Deeper CNN Accuracy: {cnn6_accuracy:.2f}%\n')

Training Deeper CNN...
Epoch 1/5, Loss: 0.3076
Epoch 2/5, Loss: 0.1023
Epoch 3/5, Loss: 0.0789
Epoch 4/5, Loss: 0.0593
Epoch 5/5, Loss: 0.0538
Deeper CNN Accuracy: 98.85%

